# Asbestos Screening Tool (Google Colab)
**Important:** this tool *screens* drawings and material lists for materials that may contain asbestos. It cannot prove asbestos is absent. Presence or absence is confirmed only by laboratory analysis of a sample taken by a licensed professional.

Run the cells **in order, top to bottom**. When Cell 6 prints a link, open it.

### Cell 1: Install packages

In [ ]:
%pip install -q streamlit easyocr opencv-python-headless rapidfuzz pandas pillow openpyxl

### Cell 2: Reference dataset (43 material types, 356 search names)

In [ ]:
%%writefile asbestos_reference.csv
ref_id,material_name,aliases,category,typical_locations,friable,risk_level,notes
M001,Material explicitly labelled asbestos,asbestos|acm|asbestos containing material|asbestos containing|contains asbestos|asbestos warning,Explicit label,Anywhere it is marked on drawings or schedules,Possible,High,Drawing or schedule itself says asbestos. Treat as asbestos-containing until proven otherwise.
M002,Asbestos cement (AC) sheet - roofing and cladding,asbestos cement sheet|ac sheet|a c sheet|ac sheets|asbestos sheet|corrugated cement sheet|corrugated asbestos sheet|ac roofing|fibro sheet|fibro|cement roof sheet,Asbestos cement,"Roofs, sheds, factory and godown roofs, wall cladding, partitions",No,High,"Very common in older buildings. Non-friable: releases fibres when cut, drilled, broken or weathered. In some countries (including India) chrysotile AC sheets are still manufactured, so newer buildings are not automatically free of it."
M003,Asbestos cement (AC) pipes,asbestos cement pipe|ac pipe|ac pressure pipe|ac drain pipe|ac downpipe|ac gutter|ac water main|cement pipe|cement drain pipe,Asbestos cement,"Water mains, drains, soil and rainwater pipes, underground services",No,High,"Non-friable: releases fibres when cut, drilled, broken or weathered."
M004,"Rainwater goods (gutters, downpipes)",gutter|gutters|downpipe|down pipe|rainwater goods|rainwater pipe|rwp|soil pipe|waste pipe|drain pipe|sewer pipe,Asbestos cement,External walls and roof edges,No,Medium,Older rainwater goods were often asbestos cement. Material type must be confirmed.
M005,Sprayed insulation / fireproofing,sprayed insulation|sprayed coating|sprayed fireproofing|spray applied insulation|spray fireproofing|fireproofing spray|limpet|sprayed asbestos,Thermal / fire protection,"Steel beams, columns, ceilings, soffits, plant rooms",Yes,High,Friable: crumbles easily and releases fibres readily. Do not disturb. One of the highest-risk materials.
M006,Pipe insulation / lagging,pipe insulation|pipe lagging|pipe covering|pipe wrap|thermal pipe insulation|steam pipe insulation|lagging,Thermal insulation,"Heating, steam and hot-water pipework, risers, plant rooms",Yes,High,Friable: crumbles easily and releases fibres readily. Do not disturb.
M007,Boiler / tank / furnace insulation,boiler insulation|boiler lagging|calorifier insulation|hot water tank insulation|tank lagging|furnace insulation|kiln lining|furnace lining,Thermal insulation,"Boiler rooms, plant rooms, industrial furnaces",Yes,High,Friable: crumbles easily and releases fibres readily. Do not disturb.
M008,Duct insulation and flexible connectors,duct insulation|hvac duct insulation|duct wrap|flexible duct connector|duct connector,Thermal insulation,"Air-handling units, HVAC ducts",Possible,Medium,Canvas connectors and duct wraps in older systems may contain asbestos.
M009,Asbestos insulating board (AIB) / fire board,asbestos insulating board|aib|asbestos board|insulating board|fire board|fireproof board|fire resistant board|fire resistant panel|fire barrier board,Boards and panels,"Fire doors, ceiling and wall linings, service riser covers, partitions",Possible,High,Softer than cement sheet and releases fibres more easily. Common in 1950s-1980s buildings.
M010,"Soffit, ceiling and partition board",soffit board|soffit|ceiling board|partition board|wall board|panel board|fascia|eaves|bargeboard|barge board,Boards and panels,"Eaves, porches, ceilings, partitions, balcony undersides",No,Medium,"Board type must be confirmed. Non-friable: releases fibres when cut, drilled, broken or weathered."
M011,Vermiculite loose-fill insulation,vermiculite|vermiculite insulation|loose fill insulation|loose fill,Thermal insulation,"Lofts, wall cavities, ceiling voids",Yes,High,Some vermiculite was contaminated with asbestos. Friable: crumbles easily and releases fibres readily. Do not disturb.
M012,Loft / attic / ceiling insulation (type unknown),loft insulation|attic insulation|ceiling insulation|cavity insulation|insulation blanket|insulation mattress,Thermal insulation,"Lofts, ceilings, wall cavities",Possible,Medium,Most modern insulation is mineral wool or foam; older material must be checked.
M013,General insulation (type unknown),insulation|insulating material|insulation material|thermal insulation|heat insulation,Thermal insulation,Anywhere,Possible,Medium,Type is not stated. Identify the material and age before assuming it is safe.
M014,Ceiling tiles / acoustic ceilings,ceiling tile|ceiling tiles|acoustic tile|acoustic ceiling|suspended ceiling|false ceiling|drop ceiling|ceiling panel,Ceilings,"Offices, schools, hospitals, corridors",No,Medium,"Older acoustic and fibre ceiling tiles may contain asbestos. Gypsum or PVC false ceilings usually do not, but verify."
M015,Textured coatings / popcorn ceilings,textured coating|textured paint|artex|stipple ceiling|popcorn ceiling|textured ceiling|spray texture|stipple,Coatings,Ceilings and walls in older homes,No,Medium,Textured decorative coatings sold before bans often contained asbestos.
M016,Vinyl floor tiles and sheet flooring,vinyl floor tile|vinyl tile|vinyl flooring|vinyl floor|vinyl asbestos tile|vat|asphalt tile|thermoplastic tile|resilient flooring|sheet vinyl|vinyl sheet|linoleum|lino|floor vinyl,Flooring,"Floors in homes, schools, offices, hospitals",No,Medium,Older 9x9 inch and 12x12 inch tiles and their backing are commonly suspect.
M017,Floor / tile adhesive and mastic,floor adhesive|tile adhesive|mastic|black mastic|bituminous adhesive|flooring adhesive|cutback adhesive|adhesive,Flooring,"Under vinyl, parquet and tiles",No,Medium,Black bituminous adhesives under old floors can contain asbestos even if the tile does not.
M018,Floor tile (type unknown),floor tile|floor tiles|flooring tile|flooring,Flooring,Floors,No,Low-possible,"Ceramic and stone tiles normally do not contain asbestos; backing, screed and adhesive may."
M019,Wall or bathroom tile (type unknown),bathroom tile|wall tile|ceramic tile|tile|tiles|tiling,Flooring / walls,"Bathrooms, kitchens",No,Low-possible,"The tile itself is rarely the issue; check adhesive, grout and backing board."
M020,Cement board / flat cement sheet,cement board|cement sheet|flat sheet|fibre cement board|fibre cement|fiber cement|cement panel|cement fibre|compressed sheet|building board,Asbestos cement,"Wall linings, bathrooms, internal partitions",No,Medium,"Pre-ban fibre cement often contained asbestos. Non-friable: releases fibres when cut, drilled, broken or weathered."
M021,External cladding / weatherboard,external cladding|cladding panel|cladding|wall cladding|weatherboard|cement cladding|asbestos cladding,Asbestos cement,External walls,No,Medium,"Non-friable: releases fibres when cut, drilled, broken or weathered."
M022,Wall and partition panels,wall panel|wall panels|partition panel|partition wall|panel wall|sandwich panel|prefab panel|prefabricated panel,Boards and panels,"Walls, partitions, prefab buildings",Possible,Medium,Panel core and skin material must be identified.
M023,Roofing felt / bituminous sheet / membranes,roofing felt|bitumen felt|bituminous felt|bitumen sheet|bituminous sheet|roof membrane|built up roofing|roof flashing|roof coating|roof sealant|waterproofing membrane|damp proof course|dpc,Roofing,"Flat roofs, parapets, damp-proofing",No,Medium,Older bituminous products sometimes contained asbestos fibres.
M024,Roof or wall sheet (type unknown),roof sheet|roofing sheet|corrugated sheet|roofing|roof,Roofing,Roofs,No,Medium,"Many roof sheets are metal, but older fibre-cement ones are asbestos. Identify the type."
M025,Roof tiles / slates / shingles,roof tile|roof tiles|shingles|roof shingle|slate roof|cement slate|fibre cement slate|ridge cap|ridge tile,Roofing,Pitched roofs,No,Low-possible,Clay and concrete tiles are normally free of asbestos; cement slates and some shingles are not.
M026,Water tanks and cisterns,water tank|cistern|cold water tank|ac cistern|asbestos cistern|water storage tank|overhead tank,Asbestos cement,"Roofs, loft spaces, tank rooms",No,Medium,Older AC water tanks exist. Plastic and steel tanks do not contain asbestos.
M027,"Flues, chimneys and exhaust pipes",flue pipe|chimney|chimney flue|flue|exhaust flue|vent pipe|flue liner|stack pipe,Asbestos cement,"Boilers, heaters, kitchens, generators",No,Medium,AC flue pipes and liners are common in older installations.
M028,"Rope, cloth, paper and tape",asbestos rope|insulating rope|woven tape|asbestos cloth|asbestos paper|glass tape|rope seal|door seal|furnace door seal|fire door rope,Textiles / seals,"Boilers, furnaces, ovens, fire doors",Yes,High,Friable: crumbles easily and releases fibres readily. Do not disturb.
M029,Gaskets and packing,gasket|gaskets|flange gasket|joint packing|packing|boiler door,Textiles / seals,"Pipe flanges, boilers, pumps, valves",Possible,Medium,Compressed asbestos gaskets were widely used on steam and chemical lines.
M030,"Fire doors, fire shutters and blankets",fire door|fire rated door|fire door core|fire shutter|fire curtain|fire blanket|fire resistant door|fire check door|safety curtain|welding blanket|hot work blanket|heat resistant mat,Fire protection,"Corridors, stairwells, workshops, theatres",Possible,Medium,"Cores, linings and seals of older fire products may contain asbestos."
M031,Switchboards and electrical backing,switchboard|fuse board|fuse box|meter board|arc chute|switch gear|switchgear|panel backing|insulating panel,Electrical,"Substations, meter rooms, plant rooms",No,Medium,Older switchgear used asbestos arc shields and backing boards.
M032,Electrical and distribution panels,electrical panel|distribution board|electrical board|control panel|cable duct|cable trough|cable tray|cement duct|conduit,Electrical,Electrical rooms and risers,No,Low-possible,Cement cable ducts and troughs in older sites can be asbestos cement.
M033,"Plaster, render and joint compounds",plaster|wall plaster|skim coat|joint compound|jointing compound|drywall joint|gypsum plaster|render|cement render|stucco|plasterboard|plaster board|gypsum board|drywall,Plaster / finishes,Walls and ceilings,No,Low-possible,"Mostly asbestos-free, but joint compounds and textured plasters made before bans sometimes contained it."
M034,"Putty, caulking and sealants",window putty|glazing putty|glazing compound|caulk|caulking|sealant|sealing compound|expansion joint|expansion joint filler|bituminous sealant|mastic sealant,Sealants,"Windows, expansion joints, around pipes and ducts",No,Medium,Older glazing putty and sealants often contained asbestos.
M035,Bituminous and heat-resistant paints,bitumen paint|bituminous paint|anti condensation|underseal|heat resistant paint|fire retardant paint|intumescent paint|fireproof paint|fire protective coating,Coatings,"Metalwork, pipes, roofs, steel",No,Low-possible,Some older protective coatings contained asbestos fillers.
M036,Laboratory benches and fume cupboards,fume cupboard|fume hood|laboratory bench|lab bench|bench top|lab worktop|lab table|acid resistant bench,Laboratory,School and industrial laboratories,No,Medium,Older lab tops and fume cupboard linings were often asbestos cement or board.
M037,Cooling tower fill and louvres,cooling tower|cooling tower fill|cooling tower louvre,Asbestos cement,Industrial and HVAC plants,No,Medium,AC fill and louvres were common in older towers.
M038,Acoustic insulation and sprays,acoustic insulation|sound insulation|acoustic panel|acoustic lining|acoustic plaster|acoustic spray,Thermal / acoustic,"Auditoria, studios, plant rooms",Possible,Medium,Sprayed acoustic finishes in older buildings need testing.
M039,"Service ducts, riser covers and access panels",service duct|riser cover|access panel|inspection panel|hatch|loft hatch|ceiling hatch|access hatch|boxing|boxing in|bulkhead|pipe duct|pipe riser|pipe chase|riser,Boards and panels,"Service shafts and risers, bathrooms, corridors",Possible,Medium,Covers and boxing were often AIB. Check any lagging behind them.
M040,Pipework (insulation unknown),steam pipe|hot water pipe|heating pipe|chilled water pipe|pipework|pipe,Thermal insulation,"Plant rooms, ceilings, risers",Possible,Low-possible,Pipes themselves are not the concern; look for lagging on them.
M041,"Plant, boilers and heating appliances",boiler|boiler room|plant room|boiler house|furnace|oven|kiln|heater|water heater|calorifier,Thermal insulation,"Plant rooms, kitchens, industrial buildings",Possible,Low-possible,"Look for lagging, gaskets, rope seals and insulating board around the equipment."
M042,"Asphalt and bituminous floors, screeds",asphalt floor|bitumen floor|bituminous floor|screed|floor screed|levelling compound|floor leveling,Flooring,"Ground floors, industrial floors",No,Low-possible,Some old floor finishes contained asbestos fibres.
M043,Lift and elevator machinery,elevator brake|lift brake|lift machine room|lift motor room|machine room|elevator|lift,Mechanical,Lift shafts and motor rooms,No,Low-possible,Older brake linings and arc shields may contain asbestos.

### Cell 3: Screening logic

In [ ]:
%%writefile screening.py
"""Asbestos SCREENING logic (not a detector).
Matches text from drawings / schedules to a reference list of materials that
are known to contain asbestos in some products. Presence is only ever
confirmed by laboratory analysis of a sample."""
import re
import pandas as pd
from rapidfuzz import fuzz

RISK_RANK = {"High": 3, "Medium": 2, "Low-possible": 1}
LAB_OPTIONS = ["Pending", "Asbestos detected", "Not detected"]
MATCH_THRESHOLD = 85

# Approximate year of the national ban (None = no complete ban / not applied).
# Verify against your local regulator before relying on this.
COUNTRY_BAN_YEAR = {
    "India": None,
    "United Kingdom": 1999,
    "European Union": 2005,
    "Australia": 2003,
    "Canada": 2018,
    "United States": None,
    "Other / unknown": None,
}


def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


def load_reference(path="asbestos_reference.csv"):
    ref = pd.read_csv(path).fillna("")
    entries = []
    for _, r in ref.iterrows():
        for alias in str(r["aliases"]).split("|"):
            a = normalize(alias)
            if a:
                entries.append({
                    "alias": a,
                    "ref_id": r["ref_id"],
                    "material_name": r["material_name"],
                    "category": r["category"],
                    "typical_locations": r["typical_locations"],
                    "friable": r["friable"],
                    "risk_level": r["risk_level"],
                    "notes": r["notes"],
                })
    return ref, entries


def alias_score(q, alias):
    if re.search(rf"\b{re.escape(alias)}\b", q):
        return 100.0
    score = fuzz.ratio(q, alias)
    if len(alias) >= 9 and len(q) >= len(alias):
        score = max(score, fuzz.partial_ratio(q, alias))
    return float(score)


def match_text(text, entries, threshold=MATCH_THRESHOLD):
    q = normalize(text)
    if len(q) < 3:
        return None
    best, best_key = None, None
    for e in entries:
        s = alias_score(q, e["alias"])
        if s < threshold:
            continue
        key = (len(e["alias"]), s, RISK_RANK[e["risk_level"]])  # most specific alias wins
        if best_key is None or key > best_key:
            best, best_key = dict(e, score=round(s, 1)), key
    return best


def era_note(year, country):
    if not year:
        return "Construction year not given."
    ban = COUNTRY_BAN_YEAR.get(country)
    if ban is None:
        return (f"Built {year}: no complete national ban applied for {country}; "
                "construction year does NOT reduce the screening risk.")
    if year >= ban:
        return (f"Built {year}: after the ~{ban} ban, so likelihood is lower, but "
                "old stock, imports and reused materials are still possible. Verify.")
    return f"Built {year}: before the ~{ban} ban, when asbestos use was legal and common."


def final_status(risk, lab):
    if lab == "Asbestos detected":
        return "CONFIRMED - asbestos detected by laboratory"
    if lab == "Not detected":
        return "Not detected in the tested sample (applies only to that sample)"
    return {
        "High": "HIGH suspicion - assume asbestos until tested",
        "Medium": "POSSIBLE - sampling recommended",
        "Low-possible": "LOW-POSSIBLE - verify material type",
    }[risk]


def screen(items, entries, country="India", year=0):
    """items: list of (text, ocr_confidence, source). Returns (matched_df, unmatched_df)."""
    matched, unmatched, seen = [], [], set()
    for text, conf, source in items:
        key = normalize(text)
        if not key or key in seen:
            continue
        seen.add(key)
        m = match_text(text, entries)
        if m is None:
            unmatched.append({"Text found": text, "OCR confidence": round(float(conf), 3),
                              "Source": source})
            continue
        matched.append({
            "Text found": text,
            "Source": source,
            "OCR confidence": round(float(conf), 3),
            "OCR check": "Low OCR confidence - verify text" if (source == "image" and conf < 0.4) else "OK",
            "Matched material": m["material_name"],
            "Category": m["category"],
            "Match score": m["score"],
            "Likelihood": m["risk_level"],
            "Friable": m["friable"],
            "Typical locations": m["typical_locations"],
            "Notes": m["notes"],
            "Era note": era_note(year, country),
            "Lab result": "Pending",
        })
    cols_m = ["Text found","Source","OCR confidence","OCR check","Matched material","Category",
              "Match score","Likelihood","Friable","Typical locations","Notes","Era note","Lab result"]
    mdf = pd.DataFrame(matched, columns=cols_m)
    if len(mdf):
        mdf["_r"] = mdf["Likelihood"].map(RISK_RANK)
        mdf = mdf.sort_values("_r", ascending=False).drop(columns="_r").reset_index(drop=True)
    udf = pd.DataFrame(unmatched, columns=["Text found", "OCR confidence", "Source"])
    return mdf, udf


def add_status(mdf):
    mdf = mdf.copy()
    mdf["Screening status"] = [final_status(r, l) for r, l in zip(mdf["Likelihood"], mdf["Lab result"])]
    return mdf

### Cell 4: The web app

In [ ]:
%%writefile app.py
import io
import cv2
import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image

from screening import (load_reference, screen, add_status, COUNTRY_BAN_YEAR,
                       LAB_OPTIONS, RISK_RANK)

st.set_page_config(page_title="Asbestos Screening Tool", layout="wide")
st.title("Asbestos Screening Tool for Building Drawings")
st.warning(
    "This tool SCREENS for materials that may contain asbestos. It cannot tell you "
    "that asbestos is absent. Presence or absence is confirmed only by laboratory "
    "analysis of a sample taken by a trained, licensed professional. Do not drill, "
    "cut, break or sample suspect material yourself."
)

ref, entries = load_reference("asbestos_reference.csv")


@st.cache_resource
def load_reader():
    import easyocr
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    return easyocr.Reader(["en"], gpu=gpu)


def ocr_image(pil_img, also_enhanced=True):
    reader = load_reader()
    arr = np.array(pil_img.convert("RGB"))
    found = [(t, c) for _, t, c in reader.readtext(arr)]
    if also_enhanced:
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
        found += [(t, c) for _, t, c in reader.readtext(gray)]
    return found


# ---------------- Sidebar ----------------
with st.sidebar:
    st.header("Building details")
    country = st.selectbox("Country / region", list(COUNTRY_BAN_YEAR.keys()))
    year = st.number_input("Year of construction (0 = unknown)", 0, 2100, 0, 1)
    st.caption("Era rules are approximate. Confirm with your local regulator.")
    st.divider()
    with st.expander("Reference dataset"):
        st.dataframe(ref[["ref_id", "material_name", "category", "risk_level", "friable"]],
                     hide_index=True)

# ---------------- Inputs ----------------
col1, col2 = st.columns(2)
with col1:
    images = st.file_uploader("Upload drawings / photos of material schedules",
                              type=["png", "jpg", "jpeg"], accept_multiple_files=True)
    enhance = st.checkbox("Also scan a 2x enhanced copy (slower, finds more text)", True)
with col2:
    typed = st.text_area(
        "Or type / paste materials (one per line)",
        placeholder="Roof: AC sheet\nPipe insulation in boiler room\nCeiling tiles 600x600\nVinyl floor tile",
        height=180)
    sched = st.file_uploader("Or upload a material schedule (CSV / Excel)",
                             type=["csv", "xlsx"])

if st.button("Run screening", type="primary"):
    items = []
    with st.spinner("Reading inputs..."):
        for f in images or []:
            for t, c in ocr_image(Image.open(f), enhance):
                items.append((t, c, "image"))
        for line in typed.splitlines():
            if line.strip():
                items.append((line.strip(), 1.0, "typed"))
        if sched is not None:
            df_s = pd.read_csv(sched) if sched.name.lower().endswith(".csv") else pd.read_excel(sched)
            for v in df_s.astype(str).values.ravel():
                if v and v.lower() != "nan":
                    items.append((v, 1.0, "schedule"))
    if not items:
        st.error("Nothing to screen. Upload an image, paste text or upload a schedule.")
    else:
        m, u = screen(items, entries, country, int(year))
        st.session_state["matched"] = m
        st.session_state["unmatched"] = u

# ---------------- Results ----------------
if "matched" in st.session_state:
    matched = st.session_state["matched"]
    unmatched = st.session_state["unmatched"]

    st.subheader("1. Suspect materials found")
    if matched.empty:
        st.info("No suspect materials were recognised in the input. This is NOT an "
                "asbestos clearance. The drawing may simply not list materials.")
    else:
        st.caption("After a laboratory report arrives, set 'Lab result' for that row.")
        shown = ["Text found", "Matched material", "Likelihood", "Friable", "Lab result",
                 "OCR check", "Match score", "Typical locations", "Notes", "Era note", "Source"]
        edited = st.data_editor(
            matched[shown],
            hide_index=True,
            
            disabled=[c for c in shown if c != "Lab result"],
            column_config={"Lab result": st.column_config.SelectboxColumn(
                "Lab result", options=LAB_OPTIONS, required=True)},
            key="editor",
        )
        full = matched.copy()
        full["Lab result"] = edited["Lab result"].values
        report = add_status(full)

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("HIGH suspicion", int((report["Likelihood"] == "High").sum()))
        c2.metric("Possible / low-possible", int((report["Likelihood"] != "High").sum()))
        c3.metric("Confirmed by lab", int((report["Lab result"] == "Asbestos detected").sum()))
        c4.metric("Awaiting lab", int((report["Lab result"] == "Pending").sum()))

        st.subheader("2. Screening status")
        st.dataframe(report[["Text found", "Matched material", "Likelihood",
                             "Lab result", "Screening status"]],
                     hide_index=True)

        st.download_button("Download full report (CSV)",
                           report.to_csv(index=False).encode("utf-8"),
                           "asbestos_screening_report.csv", "text/csv")

        st.subheader("3. Next steps")
        st.markdown(
            "1. Do **not** disturb HIGH and POSSIBLE items.\n"
            "2. Ask a licensed asbestos surveyor to inspect and take samples.\n"
            "3. Send samples to an accredited laboratory.\n"
            "4. Enter the lab result above to update the status.\n"
            "5. Follow your local rules for management or removal.")

    with st.expander(f"Text that matched nothing ({len(unmatched)}) - review manually"):
        st.caption("Room names and unreadable text land here. Unmatched does not mean safe.")
        st.dataframe(unmatched, hide_index=True)

### Cell 5: Quick test (no OCR needed). Everything should match except 'Kitchen' and 'Bedroom'.

In [ ]:
from screening import *
ref, entries = load_reference()
test = ["Roof: AC sheet", "Pipe insulaton", "Ceiling tiles 600x600", "Vinyl floor tile", "Kitchen", "Bedroom"]
m, u = screen([(t, 1.0, "typed") for t in test], entries, "India", 1985)
display(add_status(m)[["Text found", "Matched material", "Likelihood", "Screening status"]])
print("Not matched:", u["Text found"].tolist())

### Cell 6: Start the app and get a public link (Cloudflare tunnel, no account needed)

In [ ]:
import subprocess, time, re, urllib.request

# stop anything left over from earlier runs
subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True)

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.address", "0.0.0.0", "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2)
        print("Streamlit is ready")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit did not start. Run  !streamlit run app.py  to see the error.")

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nOPEN THIS LINK:", m.group(0))
        break

**Using the app**
1. Pick the country and construction year in the sidebar.
2. Upload a drawing or material schedule image, paste material names, or upload a CSV/Excel schedule.
3. Click **Run screening**.
4. After a licensed surveyor's samples come back from an accredited lab, set **Lab result** per row. Only then does a row show "CONFIRMED" or "Not detected in the tested sample".

If you change any file, re-run Cells 2-4, then Cell 6. Keep this notebook running while you use the app.